# Suite SCH — Schemas

Builders, `Spec` resolution, native conversion, and on-demand `validate()` for every schema kind.
See `TestPlan.md` for the plan. Each cell is one test case, named by its ID.

In [ ]:
import base64
import math

from mbse.Schemas.Framework import Schemas as S
from mbse.Schemas.Framework.Errors import DecodeError
from support import raises, text
from mbse.Schemas.Framework import Proxies

store = Proxies.OfStore()

NATIVES = (int, float, str, bool, bytes)

## SCH-01 · `OfNative.Data` equality and validation
Supported natives validate; unsupported types and a missing type are reported. `bool` is not `int`.

In [ ]:
for native in NATIVES:
    assert S.OfNative.Data(native).validate() == [], native
for unsupported in (complex, list, dict, tuple, bytearray, type(None), object, None):
    assert S.OfNative.Data(unsupported).validate(), unsupported
assert S.OfNative.Data(str) == S.OfNative.Data(str)
assert S.OfNative.Data(bool) != S.OfNative.Data(int)
assert S.OfNative.Builder().create() == S.OfNative.Data(None)
assert S.OfNative.Builder().create().validate()

## SCH-02 · `OfNative.resolve` accepts a type, a callable, or data

In [ ]:
assert S.OfNative.resolve(str) == S.OfNative.Data(str)
assert S.OfNative.resolve(lambda n: n.type(bytes)) == S.OfNative.Data(bytes)
existing = S.OfNative.Data(int)
assert S.OfNative.resolve(existing) is existing
with raises(TypeError, match="must return its builder"):
    S.OfNative.resolve(lambda n: None)
with raises(TypeError):
    S.OfNative.resolve("str")
with raises(TypeError):
    S.OfNative.resolve(3)

## SCH-03 · `to_plain` / `from_plain`: exact types, both directions
Distinct native types are never coerced; subclasses of natives are rejected too. Writing a wrong type is a program
error (`TypeError`); reading one is a problem in the data (`DecodeError`, a `ValueError`, with no location here).

In [ ]:
cases = {int: [0, -1, 2**100, -(2**63)], float: [0.0, -0.0, 0.1, 1e308, 5e-324], str: ["", "x", "日本語 🚀", "\x00"],
         bool: [True, False], bytes: [b"", b"\x00\xff", bytes(range(256))]}
for native, values in cases.items():
    schema = S.OfNative.Data(native)
    for value in values:
        back = schema.from_plain(schema.to_plain(value))
        assert type(back) is native and back == value, (native, value)
        if native is float:
            assert math.copysign(1, back) == math.copysign(1, value)

wrong = {int: [True, 1.0, "1", None], float: [1, True, None], str: [b"x", 1, None], bool: [0, 1, "true", None],
         bytes: [bytearray(b"x"), "AA==", None]}
for native, values in wrong.items():
    for value in values:
        with raises(TypeError):
            S.OfNative.Data(native).to_plain(value)


class MyInt(int):
    pass


with raises(TypeError):
    S.OfNative.Data(int).to_plain(MyInt(3))
with raises(DecodeError, match="expected int, got MyInt"):
    S.OfNative.Data(int).from_plain(MyInt(3))

for native, plain, message in [
    (int, True, "expected int, got bool"),
    (int, 1.0, "expected int, got float"),
    (float, 1, "expected float, got int"),
    (bool, 0, "expected bool, got int"),
    (str, b"x", "expected str, got bytes"),
    (str, None, "expected str, got NoneType"),
    (bytes, b"raw", "expected base64 text for bytes, got bytes"),
]:
    try:
        S.OfNative.Data(native).from_plain(plain)
    except DecodeError as error:
        assert str(error) == error.reason == message and (error.line, error.column, error.path) == (None, None, None)
    else:
        raise AssertionError((native, plain))

## SCH-04 · Bytes are strict base64 text

In [ ]:
schema = S.OfNative.Data(bytes)
assert schema.to_plain(b"\x00\xff") == "AP8="
assert schema.from_plain(base64.b64encode(b"hello").decode()) == b"hello"
for bad in ["AP8", "AP8=\n", "A P8=", "not base64!", "AP8=="]:
    with raises(DecodeError, match="invalid base64 text"):
        schema.from_plain(bad)

## SCH-05 · Non-finite floats have exactly three plain forms

In [ ]:
schema = S.OfNative.Data(float)
assert schema.to_plain(math.inf) == "Infinity" and schema.to_plain(-math.inf) == "-Infinity"
assert schema.to_plain(math.nan) == "NaN" and schema.to_plain(float("-nan")) == "NaN"
assert schema.from_plain("Infinity") == math.inf and schema.from_plain("-Infinity") == -math.inf
assert math.isnan(schema.from_plain("NaN"))
for bad in ["nan", "inf", "+Infinity", "infinity", "NAN", "1.5", ""]:
    with raises(DecodeError, match=f"expected a float or one of ['-Infinity', 'Infinity', 'NaN'], got {bad!r}"):
        schema.from_plain(bad)
with raises(DecodeError, match="expected str, got float"):
    S.OfNative.Data(str).from_plain(1.5)  # strings are only special for float

## SCH-06 · `OfAny.Builder`: selecting a kind yields that kind's data

In [ ]:
assert type(S.OfAny.Builder().as_native(str).create()) is S.OfNative.Data
obj = S.OfAny.Builder().as_object(lambda o: o.properties(text("x"))).create()
assert type(obj) is S.OfObject.Data and list(obj.properties) == ["x"]
assert S.OfAny.Builder().as_native(str).as_native(int).create() == S.OfNative.Data(int)  # the last selection wins
with raises(ValueError, match="no kind selected"):
    S.OfAny.Builder().create()

## SCH-07 · `OfAny.Builder` with a source: create / clone / update

In [ ]:
source = S.OfNative.Data(str)
with raises(ValueError):
    S.OfAny.Builder(source).as_native(int).create()
copy_ = S.OfAny.Builder(source).clone()
assert copy_ == source and copy_ is not source
assert S.OfAny.Builder(source).as_native(int).clone() == S.OfNative.Data(int) and source == S.OfNative.Data(str)
assert S.OfAny.Builder(source).as_native(bytes).update() is source and source.type is bytes
with raises(TypeError, match="cannot change the kind"):
    S.OfAny.Builder(source).as_object(lambda o: o).update()
with raises(ValueError):
    S.OfAny.Builder(source).update()  # nothing selected
with raises(ValueError):
    S.OfAny.Builder().clone()
with raises(ValueError):
    S.OfAny.Builder().as_native(str).update()

## SCH-08 · `OfAny.resolve` accepts value kinds only

In [ ]:
for data in [S.OfNative.Data(str), S.OfObject.Data(), S.OfUnion.Data(), S.OfIntersection.Data()]:
    assert S.OfAny.resolve(data) is data
relation = S.OfRelation.Builder().links("a", "b").create()
with raises(TypeError):
    S.OfAny.resolve(relation)
with raises(TypeError):
    S.OfObject.Builder().properties(lambda p: p.name("r").of(relation)).create()
with raises(TypeError):
    S.OfAny.resolve(42)

## SCH-09 · Finalization rules hold for every builder

In [ ]:
builders = [
    (S.OfNative.Builder, S.OfNative.Data(str)),
    (S.OfObject.Builder, S.OfObject.Data()),
    (S.OfRelation.Builder, S.OfRelation.Data()),
    (S.OfAdjacency.Builder, S.OfAdjacency.Data()),
    (S.OfUnion.Builder, S.OfUnion.Data()),
    (S.OfIntersection.Builder, S.OfIntersection.Data()),
    (S.OfProperty.Builder, S.OfProperty.Data()),
]
for Builder, source in builders:
    assert type(Builder().create()) is type(source)
    with raises(ValueError, match="only valid without a source"):
        Builder(source).create()
    with raises(ValueError, match="only valid with a source"):
        Builder().clone()
    with raises(ValueError, match="only valid with a source"):
        Builder().update()
    cloned = Builder(source).clone()
    assert cloned is not source and type(cloned) is type(source)
    assert Builder(source).update() is source

## SCH-10 · Builders copy their own containers, never referenced schemas

In [ ]:
Rel = S.OfRelation.Builder().links("a", "b").create()
Base = S.OfObject.Builder().properties(text("x")).relations(lambda a: a.name("r").of(Rel).me("a")).create()

extended = S.OfObject.Builder(Base).properties(text("y")).clone()
assert list(Base.properties) == ["x"] and list(extended.properties) == ["x", "y"]
assert extended.properties is not Base.properties and extended.adjacencies is not Base.adjacencies
assert extended.adjacencies["r"] is Base.adjacencies["r"]  # adjacency data is shared, not copied
assert extended.adjacencies["r"].relation is Rel  # references keep identity

builder = S.OfObject.Builder(Base).properties(text("z"))
assert "z" not in Base.properties  # nothing reaches the source before update()
first = builder.clone()
builder.properties(text("w"))
second = builder.clone()
assert "w" not in first.properties and "w" in second.properties

same = S.OfObject.Builder(Base).properties(text("v")).update()
assert same is Base and "v" in Base.properties

## SCH-11 · `OfObject`: properties and relations are keyed by name, in order

In [ ]:
o = (S.OfObject.Builder()
     .properties(text("b"), text("a"), text("b", int))
     .properties(text("c"))
     .create())
assert list(o.properties) == ["b", "a", "c"] and o.properties["b"] == S.OfNative.Data(int)
Rel2 = S.OfRelation.Builder().links("x", "y").create()
o2 = S.OfObject.Builder().relations(lambda a: a.name("r").of(Rel).me("a"), lambda a: a.name("r").of(Rel2).me("x")).create()
assert list(o2.adjacencies) == ["r"] and o2.adjacencies["r"].relation is Rel2
assert S.OfObject.Builder().singleton("Global.Name").create().singleton == "Global.Name"
assert S.OfObject.Data().validate() == []

## SCH-12 · `OfObject.validate` reports clashes and nested problems

In [ ]:
clash = S.OfObject.Builder().properties(text("r")).relations(lambda a: a.name("r").of(Rel).me("a")).create()
assert any("both property and adjacency" in p for p in clash.validate())
nested = S.OfObject.Builder().properties(text("bad", list)).create()
assert nested.validate() == ["property 'bad': unsupported native type <class 'list'>"]
wrong_side = S.OfObject.Builder().relations(lambda a: a.name("r").of(Rel).me("c")).create()
assert any("'c' is not a link" in p for p in wrong_side.validate())
no_relation = S.OfObject.Builder().relations(lambda a: a.name("r").me("a")).create()
assert any("has no relation" in p for p in no_relation.validate())

## SCH-13 · `OfRelation.validate` covers link, property and uniqueness problems

In [ ]:
def problems(builder):
    return builder.create().validate()


assert problems(S.OfRelation.Builder().links("a", "b")) == []
assert any("one-link" in p for p in problems(S.OfRelation.Builder().links("a")))
assert any("at least two links" in p for p in problems(S.OfRelation.Builder()))
assert any("duplicate link" in p for p in problems(S.OfRelation.Builder().links("a", "a")))
assert any("both link and property" in p for p in problems(S.OfRelation.Builder().links("a", "b").properties(text("a"))))
assert any("unknown" in p for p in problems(S.OfRelation.Builder().links("a", "b").unique("c")))
assert problems(S.OfRelation.Builder().links("a", "b").unique()) == []  # unique() is trivially true
assert problems(S.OfRelation.Builder().links("a", "b").properties(text("k")).unique("a", "b", "k")) == []
assert any("property 'k'" in p for p in problems(S.OfRelation.Builder().links("a", "b").properties(text("k", list))))
assert S.OfRelation.Builder().links("a", "b").links("c", "d").create().links == ("c", "d")  # links() replaces
r = S.OfRelation.Builder().links("a", "b").unique("a").unique("b").unique("a").create()
assert r.uniques == (frozenset({"a"}), frozenset({"b"}), frozenset({"a"}))  # clauses accumulate as written

## SCH-14 · Unions: named branches accumulate in order; validation checks count, kinds and names

In [ ]:
A = S.OfObject.Builder().properties(text("a")).create()
B = S.OfObject.Builder().properties(text("b")).create()
u = S.OfUnion.Builder().branches(lambda b: b.name("p").of(A)).branches(lambda b: b.name("q").of(B)).create()
assert [b.type for b in u.branches] == [A, B] and [b.name for b in u.branches] == ["p", "q"]
assert u.properties == {"p": A, "q": B} and type(u.branches[0]) is S.OfUnion.Branch
assert u.validate() == []
assert S.OfUnion.Builder().branches(lambda b: b.name("p").of(A)).create().validate() == [
    "a union needs at least two branches"]
native = lambda t: t.as_native(str)
assert S.OfUnion.Builder().branches(lambda b: b.name("p").of(A), lambda b: b.name("q").of(native)).create().validate() == [
    "union branches must all be the same kind"]
with raises(TypeError, match="as_native(str)"):
    S.OfUnion.Builder().branches(lambda b: b.of(str))  # a bare class is not an OfAny.Spec
with raises(TypeError, match="must return its builder"):
    S.OfObject.Builder().properties(lambda p: "not a builder")
unnamed = S.OfUnion.Builder().branches(lambda b: b.of(A), lambda b: b.name("q").of(B)).create().validate()
assert unnamed == ["branch 0 has no name"]
twice = S.OfUnion.Builder().branches(lambda b: b.name("p").of(A), lambda b: b.name("p").of(B)).create().validate()
assert twice == ["branch name 'p' is used more than once"]

## SCH-15 · Intersections: named parts accumulate; parts may declare the same property with different types

In [ ]:
X1 = S.OfObject.Builder().properties(text("x")).create()
X2 = S.OfObject.Builder().properties(lambda p: p.name("x").of(lambda t: t.as_native(str)), text("y")).create()
X3 = S.OfObject.Builder().properties(text("x", int)).create()
x1, x2, x3 = (lambda p: p.name("x1").of(X1)), (lambda p: p.name("x2").of(X2)), (lambda p: p.name("x3").of(X3))
i = S.OfIntersection.Builder().parts(x1).parts(x2).create()
assert [p.type for p in i.parts] == [X1, X2] and i.properties == {"x1": X1, "x2": X2}
assert type(i.parts[0]) is S.OfIntersection.Part
assert S.OfIntersection.Builder().parts(x1, x2, x3).create().validate() == []  # each part keeps its own 'x'
assert S.OfIntersection.Builder().parts(x1).create().validate() == ["an intersection needs at least two parts"]
assert S.OfIntersection.Builder().parts(x1, lambda p: p.name("s").of(S.OfNative.Data(str))).create().validate() == [
    "intersection parts must all be the same kind"]
assert S.OfIntersection.Builder().parts(x1, lambda p: p.of(X2), x1).create().validate() == [
    "part 1 has no name", "part name 'x1' is used more than once"]

## SCH-16 · Finding F2 (pinned): `validate()` accepts names that bindings may reserve
Empty names, names with spaces or `$`, leading underscores, and names of builder methods all validate. Whether the
schema layer should reject them is an open question in `FRAMEWORK.md`.

In [ ]:
odd = S.OfObject.Builder().properties(*(text(n) for n in ["", "with space", "$ref", "_values", "create", "accept"])).create()
assert odd.validate() == ["name '$ref' is reserved: names starting with '$' belong to the wire format"]  # the others: F2

## SCH-17 · Union and intersection selection; identity equality; Spec errors name what they got

In [ ]:
U = S.OfAny.Builder().as_union(lambda u: u.branches(lambda b: b.name("p").of(A), lambda b: b.name("q").of(B))).create()
I = S.OfAny.Builder().as_intersection(lambda i: i.parts(x1, x2)).create()
assert type(U) is S.OfUnion.Data and len(U.branches) == 2 and type(I) is S.OfIntersection.Data and len(I.parts) == 2
assert S.OfUnion.resolve(U) is U and S.OfIntersection.resolve(I) is I
assert type(S.OfUnion.resolve(lambda u: u)) is S.OfUnion.Data and type(S.OfIntersection.resolve(lambda i: i)) is S.OfIntersection.Data
for data in [S.OfObject.Data(), S.OfRelation.Data(), S.OfAdjacency.Data(), S.OfUnion.Data(), S.OfIntersection.Data(),
             S.OfProperty.Data(), S.OfUnion.Branch()]:
    assert data == data and data != type(data)()  # schemas other than natives compare by identity
assert S.OfObject.Data(properties={"x": "junk"}).validate() == ["property 'x': not a schema: 'junk'"]
untyped = S.OfUnion.Builder().branches(lambda b: b.name("p"), lambda b: b.name("q").of(A)).create()
assert untyped.branches[0].type is None and untyped.branches[0] == untyped.branches[0]
assert "union branches must all be the same kind" in untyped.validate()


class Custom:
    pass


with raises(TypeError, match="a class is not a Spec here"):
    S.OfObject.resolve(Custom)
with raises(TypeError, match="must return its builder, got 'not a builder'"):
    S.OfObject.Builder().properties(lambda p: "not a builder")
with raises(TypeError, match="must return its builder, got <object object"):
    S.OfObject.Builder().properties(lambda p: object())
with raises(TypeError, match="must return its builder, got <function"):
    S.OfObject.Builder().properties(lambda p: (lambda: None))

## SCH-18 · Native types are tokens, with an optional width in bits or bytes
A host type is shorthand for the `basic` token of the same name. An implementation reads `basic` tokens and its own
format's (`python3`); a token in another format is valid, but describes values this implementation cannot convert.

In [ ]:
from mbse.Schemas.Framework import Comparison as C, Proxies, Validators
from support import Fake

N = S.OfNative
assert N.Data(int).token == N.Token("basic", "int") and N.Data(int).type is int and N.Data(int).host() is int
assert N.resolve(lambda n: n.type(bytes)).token == N.Token("basic", "bytes")
own = N.resolve(lambda n: n.token("python3", "int"))
assert own.type is int and own.validate() == [] and own != N.Data(int)  # the same host type, another token
assert N.resolve(lambda n: n.token("typescript5", "BigInt")).type is None  # the other implementation's format
wide = N.resolve(lambda n: n.type(int).bytes(4))
assert wide == N.Data(int, bytes=4) and wide != N.Data(int) and (wide.bits, wide.bytes) == (None, 4)
assert wide.validate() == [] and wide.to_plain(1) == 1  # a width is size only: values are not checked against it
foreign = N.resolve(lambda n: n.token("ccpp", "int32_t").bits(32))
assert str(foreign.token) == "the ccpp type 'int32_t'" and foreign.type is None and foreign.validate() == []
for call in (foreign.host, lambda: foreign.to_plain(1)):
    with raises(TypeError, match="the ccpp type 'int32_t' has no type in this implementation"):
        call()
with raises(DecodeError, match="the ccpp type 'int32_t' has no type in this implementation"):
    foreign.from_plain(1)
assert N.resolve(lambda n: n.token("basic", "long")).validate() == ["basic has no type 'long'"]
assert N.resolve(lambda n: n.token("", "x")).validate() == ["a token needs a format and a name"]
assert N.resolve(lambda n: n.type(int).bits(0).bytes(True)).validate() == [
    "a width in bits must be a positive int, got 0", "a width in bytes must be a positive int, got True",
    "a width is in bits or in bytes, not both"]

Wire = S.OfObject.Builder().name("Wire").ref().properties(lambda p: p.name("word").of(foreign)).create()
store.register(Wire)
assert Validators.Validate(store)(Wire, Fake("Wire", {"word": 1})) == [
    "Wire#0.word: the ccpp type 'int32_t' has no type in this implementation"]
with raises(TypeError, match="the ccpp type 'int32_t' has no type in this implementation"):
    C.OfNative(foreign, 1)
assert C.OfNative(own, 1).compare(C.OfNative(N.Data(int), 1)) == 0  # both hold a Python int

## SCH-19 · Reference object schemas: marked with `.ref()`, never a property's type, the only snapshot roots
A singleton's schema is a reference object schema. Only reference objects are built with a store's builders, written
as a snapshot's root, or linked in a snapshot; a value object is built through its owner.

In [ ]:
from mbse.Schemas.Framework import Plain
from mbse.Schemas.Framework.Errors import DecodeError

Thing = S.OfObject.Builder().name("Thing").ref().properties(text("name")).create()
Part = S.OfObject.Builder().name("Part").properties(text("name")).create()
assert Thing.ref and not Part.ref and Thing.validate() == [] and Part.validate() == []
assert S.OfObject.Builder().singleton("Only").create().ref
assert S.OfObject.Data(singleton="Only").validate() == ["a singleton's schema must be a reference object schema"]
held = "a reference object schema cannot be a property's type"
assert S.OfObject.Builder().properties(lambda p: p.name("thing").of(Thing)).create().validate() == [f"property 'thing': {held}"]
Either = S.OfUnion.Builder().branches(lambda b: b.name("thing").of(Thing), lambda b: b.name("part").of(Part)).create()
Both = S.OfIntersection.Builder().parts(lambda p: p.name("thing").of(Thing), lambda p: p.name("part").of(Part)).create()
for holder in (S.OfObject.Builder(), S.OfRelation.Builder().links("a", "b")):
    assert holder.properties(lambda p: p.name("e").of(Either), lambda p: p.name("i").of(Both)).create().validate() == [
        f"property 'e': {held}", f"property 'i': {held}"]

store.register(Part)
with raises(TypeError, match="'Part' is a value object schema; a value object is built through its owner"):
    store.Part()
with raises(TypeError, match="a snapshot's root must be a reference object; 'Part' is a value object schema"):
    Plain.ToPlain(store)(Part, Fake("Part", {"name": "p"}))
with raises(TypeError, match="the root schema must be a reference object schema"):
    Plain.FromPlain(store)(Part, {"root": "s0", "objects": {"s0": {}}})
store.register(Thing)
Pair = S.OfRelation.Builder().links("thing", "part").create()
S.OfObject.Builder(Thing).relations(lambda r: r.name("parts").of(Pair).me("thing")).update()
linked = {"root": "s0", "objects": {"s0": {"parts": [{"part": {"$ref": "s1", "$schema": "Part"}}]}, "s1": {}}}
with raises(DecodeError, match="$.objects.s1: 'Part' is not a reference object schema"):
    Plain.FromPlain(store)(Thing, linked)

## SCH-20 · Every kind of schema may be named: identifiers separated by dots

In [ ]:
named = [S.OfNative.resolve(lambda n: n.name("units.Meters").type(float)),
         S.OfObject.Builder().name("crm.Contact").ref().create(),
         S.OfRelation.Builder().name("crm.Knows").links("a", "b").create(),
         S.OfUnion.Builder().name("crm.Reach").branches(text("a"), text("b")).create(),
         S.OfIntersection.Builder().name("crm.Stamped").parts(text("a"), text("b")).create(),
         S.OfIndexed.Builder().name("crm.Tags").of(lambda t: t.as_native(str)).create()]
assert [s.name for s in named] == ["units.Meters", "crm.Contact", "crm.Knows", "crm.Reach", "crm.Stamped", "crm.Tags"]
assert all(s.validate() == [] for s in named) and S.OfObject.Builder().create().name is None  # names are optional
for bad in ["", "crm.", ".Contact", "crm..Contact", "crm-Contact", "1crm.Contact", "crm.Contact v2"]:
    assert S.OfObject.Builder().name(bad).create().validate() == [
        f"name {bad!r} is not identifiers separated by dots, e.g. 'crm.Contact'"], bad
assert named[0] != S.OfNative.Data(float)  # a native's name is part of it
renamed = S.OfObject.Builder(named[1]).name("crm.Person").clone()
assert renamed.name == "crm.Person" and named[1].name == "crm.Contact" and renamed.ref